In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
ACC102 Mini Assignment – Track 2
Profitability Analyzer
Author: dzy1174
Date: 26 April 2026
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

TICKER = "AAPL"

data = {
    "Total Revenue": [347200000000, 365800000000, 394300000000],
    "Cost Of Revenue": [218200000000, 228900000000, 248900000000],
    "Operating Income": [109000000000, 119700000000, 129800000000],
    "Net Income": [94700000000, 100600000000, 112200000000],
    "Total Assets": [352600000000, 335000000000, 351100000000]
}
years = [2022, 2023, 2024]
income_df = pd.DataFrame(data, index=years).T

latest = income_df.iloc[:, -1]

revenue = latest["Total Revenue"]
cogs = latest["Cost Of Revenue"]
op_inc = latest["Operating Income"]
net_inc = latest["Net Income"]
assets = latest["Total Assets"]

gross_margin = (revenue - cogs) / revenue * 100
operating_margin = op_inc / revenue * 100
net_margin = net_inc / revenue * 100
roa = net_inc / assets * 100

latest_ratios = pd.DataFrame({
    'Metric': ['Gross Margin (%)', 'Operating Margin (%)', 
               'Net Profit Margin (%)', 'Return on Assets (%)'],
    'Value': [gross_margin, operating_margin, net_margin, roa]
})

print("Latest Year Profitability:")
print(latest_ratios.round(2))

trend_df = pd.DataFrame({
    "Year": years,
    "Gross Margin (%)": [(income_df[y]["Total Revenue"] - income_df[y]["Cost Of Revenue"]) / income_df[y]["Total Revenue"] * 100 for y in years],
    "Operating Margin (%)": [income_df[y]["Operating Income"] / income_df[y]["Total Revenue"] * 100 for y in years],
    "Net Profit Margin (%)": [income_df[y]["Net Income"] / income_df[y]["Total Revenue"] * 100 for y in years],
    "ROA (%)": [income_df[y]["Net Income"] / income_df[y]["Total Assets"] * 100 for y in years]
})

print("\nTrend (last 3 years):")
print(trend_df.round(2))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(trend_df['Year'], trend_df['Gross Margin (%)'], 'o-', label='Gross Margin')
axes[0,0].plot(trend_df['Year'], trend_df['Operating Margin (%)'], 's-', label='Operating Margin')
axes[0,0].set_title('Gross vs Operating Margin')
axes[0,0].legend()
axes[0,0].grid(True)

axes[0,1].bar(trend_df['Year'], trend_df['Net Profit Margin (%)'], color='seagreen')
axes[0,1].set_title('Net Profit Margin')
axes[0,1].grid(axis='y')

axes[1,0].plot(trend_df['Year'], trend_df['ROA (%)'], '^-', color='crimson', label='ROA')
axes[1,0].axhline(10, color='gray', linestyle='--')
axes[1,0].set_title('Return on Assets (ROA)')
axes[1,0].legend()
axes[1,0].grid(True)

axes[1,1].barh(latest_ratios['Metric'], latest_ratios['Value'], color='steelblue')
axes[1,1].set_title('Latest Year Ratios')
axes[1,1].grid(axis='x')

plt.tight_layout()
plt.savefig('outputs/profitability_dashboard.png', dpi=150)
plt.show()

latest_ratios.to_csv('data/processed/latest_ratios.csv', index=False)
trend_df.to_csv('data/processed/trend_ratios.csv', index=False)

with pd.ExcelWriter('outputs/profitability_report.xlsx') as writer:
    trend_df.to_excel(writer, sheet_name='Trends', index=False)
    latest_ratios.to_excel(writer, sheet_name='Latest Ratios', index=False)